# CDM customer fact — decode the coded columns

Turns the customer fact's keys into readable values through the CDM dimension tables, on the latest snapshot.

Cells
1. Setup and config (decode list: fact key columns → dimension, key, label columns)
2. Inputs — fact snapshot (doubled loads de-duplicated), each dimension's rows, keys and version history; current version picked per key (active, not purged, open end date, newest effective / update date)
3. Decoded values — PD grade now and in each history column side by side (with direct hard exposure, 12-month revenue and the fact's PD % as a check), industry risk (IRIS), booking platform, client tier, credit exposure type, PD-date and periodic-review age ranges; coverage of each key in its dimension
4. time_key — what it means (date columns of the time dimension) and whether it rises with as_of_dt, so other CDM tables keyed only by time_key can use it as the snapshot
5. Instrument types — flag columns of the instrument dimension (wrre_accum_flg and other _flg / _ind columns) with the instrument groups they cover, as a lead on what WRRE and the other exposure splits include

Every decode table is saved as CSV in OUT_DIR.

In [ ]:
# ============================================================
# 1. Setup and config
# ============================================================
import os, re, html, warnings, datetime as _dt
warnings.filterwarnings("ignore", category=ResourceWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore")

# --- NumPy alias shim (old pyspark code paths use removed aliases) ---
import numpy as np
for _a, _t in (("object0", object), ("bool8", np.bool_), ("int0", np.intp), ("uint0", np.uintp),
               ("bool", bool), ("object", object)):
    if not hasattr(np, _a):
        setattr(np, _a, _t)

# --- pandas shim: PySpark 3.3 createDataFrame calls iteritems (removed in pandas 2.0) ---
import pandas as pd
if not hasattr(pd.DataFrame, "iteritems"):
    pd.DataFrame.iteritems = pd.DataFrame.items
if not hasattr(pd.Series, "iteritems"):
    pd.Series.iteritems = pd.Series.items

from IPython.display import display, HTML
from pyspark.sql import SparkSession, Window, functions as F, types as T
from pyspark import StorageLevel

spark = (SparkSession.builder
         .appName("cdm_customer_fact_decode")
         .config("spark.ui.showConsoleProgress", "false")   # no stage progress bars in cell outputs
         .enableHiveSupport()
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
try:  # silence Atlas lineage hook + harmless executor-release noise (log4j2 on Spark 3.3)
    _L = spark._jvm.org.apache.logging.log4j
    for n in ("org.apache.atlas", "com.hortonworks.spark.atlas", "com.cloudera.spark.lineage",
              "org.apache.kafka", "org.apache.spark.sql.catalyst.util.package",
              "org.apache.spark.network", "io.netty"):
        _L.core.config.Configurator.setLevel(n, _L.Level.OFF)
except Exception:
    pass
# Unparseable date strings -> NULL instead of SparkUpgradeException (we probe many formats)
spark.conf.set("spark.sql.legacy.timeParserPolicy", "CORRECTED")

# ---------------- config ----------------
FACT       = "clhhd01p_clh.cdm_customer_fact_v"
CDM        = "clhhd01p_clh"
SNAP_COL   = "as_of_dt"       # bigint yyyymmdd (notebook 1)
SNAP_VALUE = "latest"         # or e.g. 20261106
ROW_KEY    = "cust_dim_key"   # one row per customer per snapshot; doubled loads are de-duplicated on it
TOP_N      = 40               # rows shown per decode table; the rest are folded into one "other" row
TIME_LOOKBACK_DAYS = 400      # snapshots used to check time_key against as_of_dt
OUT_DIR    = "../explore/cdm_decode"
os.makedirs(OUT_DIR, exist_ok=True)

# (label, fact key columns, dimension table, dimension key, label columns wanted, sort decode rows by)
# Names come from the catalog scan; label columns missing from a dimension are reported, not fatal.
DECODE = [
    ("PD grade",
     ["pd_key", "pd_prior_day_key", "pd_cd_1_mnth_key", "pd_cd_3_mnth_key", "pd_cd_1_qtr_key", "pd_cd_12_mnth_key"],
     "cdm_probability_default_dim_v", "pd_key", ["pd_cd", "pd_nbr", "pd_pct"], "pd_nbr"),
    ("Industry risk (IRIS)", ["iris_dim_key"], "cdm_iris_dim_v", "iris_dim_key",
     ["iris_cd", "iris_cd_desc", "iris_type_ind", "iris_industry_condition_cd", "iris_industry_condition_desc",
      "iris_score", "iris_direction_of_risk", "gics_sub_industry_desc"], None),
    ("Booking platform", ["platform_cd_dim_key"], "cdm_platform_code_dim_v", "platform_cd_dim_key",
     ["platform_cd", "platform_desc"], None),
    ("Client tier", ["client_strtfctn_dim_key"], "cdm_crm_client_strtfctn_dim_v", "client_strtfctn_dim_key",
     ["strtfctn_type", "smb_portfolio_flg"], None),
    ("Credit exposure type", ["cust_credit_expsr_type_dim_key"], "cdm_cust_credit_expsr_type_dim_v",
     "cust_credit_expsr_type_dim_key", ["cust_credit_expsr_type_cd", "cust_credit_expsr_type_desc"], None),
    ("Date ranges (PD date, periodic review)", ["pd_dt_data_rng_dim_key", "periodic_revw_data_rng_dim_key"],
     "cdm_data_range_dim_v", "data_range_dim_key",
     ["data_range_data_type", "range_low_val", "range_high_val", "range_sumry_desc_1", "range_sumry_desc_2",
      "range_sumry_desc_3"], "range_low_val"),
]
TIME_DIM  = "cdm_time_dim_v"              # key time_key
INSTR_DIM = "cdm_instrument_type_dim_v"   # key instrument_type_dim_key; carries wrre_accum_flg
MEASURES  = ["dhe", "gross_rev_12_mnth"]  # summed per decoded value (first fact column of each entry)

# ---------------- helpers ----------------
CSS = """<style>
.cx {font-family: Arial, sans-serif; font-size: 12px;}
.cx h3 {margin: 14px 0 4px 0; font-size: 14px;}
.cx h4 {margin: 10px 0 3px 0; font-size: 13px; color: #1f3b63;}
.cx table {border-collapse: collapse; margin: 2px 0 8px 0;}
.cx th {background: #e8eef6; border: 1px solid #c8d2e0; padding: 2px 6px; text-align: left; white-space: nowrap;}
.cx td {border: 1px solid #dde3ec; padding: 2px 6px; vertical-align: top; max-width: 420px; overflow-wrap: anywhere;}
.cx .note {color: #555; margin: 2px 0;}
.cx .warn {color: #a33; font-weight: bold;}
.cx pre {background: #f6f8fa; border: 1px solid #dde3ec; padding: 6px; white-space: pre-wrap; font-size: 11px; max-width: 1400px;}
</style>"""

def esc(x):
    return html.escape("" if x is None else str(x))

def tbl(pdf, max_rows=None):
    """pandas DataFrame -> compact HTML table (values escaped)."""
    if pdf is None or len(pdf) == 0:
        return "<p class='note'>(no rows)</p>"
    if max_rows is not None:
        pdf = pdf.head(max_rows)
    head = "".join(f"<th>{esc(c)}</th>" for c in pdf.columns)
    body = "".join("<tr>" + "".join(f"<td>{esc(v)}</td>" for v in row) + "</tr>"
                   for row in pdf.astype(object).where(pdf.notna(), "").itertuples(index=False))
    return f"<table><tr>{head}</tr>{body}</table>"

def show(parts):
    """One consolidated HTML output per cell; tex2jax_ignore keeps any dollar sign from turning into math."""
    display(HTML(CSS + "<div class='cx tex2jax_ignore mathjax_ignore'>" + "".join(parts) + "</div>"))

def fnum(x):
    if x is None:
        return ""
    try:
        x = float(x)
    except Exception:
        return str(x)
    if np.isnan(x):
        return ""
    a = abs(x)
    for d, s in ((1e12, "T"), (1e9, "B"), (1e6, "M"), (1e3, "K")):
        if a >= d:
            return f"{x / d:.2f}{s}"
    if x == int(x):
        return str(int(x))
    return f"{x:.4g}"

def fpct(x, digits=1):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return ""
    return f"{100 * float(x):.{digits}f}%"

def qcol(c):
    return F.col(f"`{c}`")

NUMERIC_T = (T.ByteType, T.ShortType, T.IntegerType, T.LongType, T.FloatType, T.DoubleType, T.DecimalType)
COMPLEX_T = (T.ArrayType, T.MapType, T.StructType, T.BinaryType)
DATE_NAME = re.compile(r"(_dt$|_dte$|_date$|^date_|_date_|_ts$|_tmstmp|timestamp|^dt_|_dt_|asof|as_of|snap|_month$|_mo$|_yrmo|yyyymm|period)")

def kind_of(dtype):
    if isinstance(dtype, (T.DateType, T.TimestampType)): return "date"
    if isinstance(dtype, T.BooleanType): return "bool"
    if isinstance(dtype, NUMERIC_T): return "num"
    if isinstance(dtype, COMPLEX_T): return "complex"
    return "str"

def is_date_like(name, dtype):
    k = kind_of(dtype)
    if k == "date":
        return True
    return k in ("str", "num") and bool(DATE_NAME.search(name.lower())) \
        and not isinstance(dtype, (T.FloatType, T.DoubleType))

def date_expr(c, dtype):
    """Parse a date-like column of any common storage format to DATE (NULL if unparseable)."""
    col = qcol(c)
    if isinstance(dtype, T.DateType):
        return col
    if isinstance(dtype, T.TimestampType):
        return F.to_date(col)
    s = F.trim(col.cast("string"))
    return (F.when(s.rlike(r"^\d{4}-\d{2}-\d{2}"), F.to_date(F.substring(s, 1, 10), "yyyy-MM-dd"))
             .when(s.rlike(r"^\d{8}$"), F.to_date(s, "yyyyMMdd"))
             .when(s.rlike(r"^\d{1,2}/\d{1,2}/\d{4}"),
                   F.to_date(F.regexp_extract(s, r"^(\d{1,2}/\d{1,2}/\d{4})", 1), "M/d/yyyy"))
             .when(s.rlike(r"^\d{6}$"), F.to_date(F.concat(s, F.lit("01")), "yyyyMMdd"))
             .when(s.rlike(r"^\d{4}-\d{2}$"), F.to_date(F.concat(s, F.lit("-01")), "yyyy-MM-dd")))

def real_date(p):
    """Drop placeholder dates (year <= 1900 or >= 2100, e.g. 0001-01-01 / 9999-12-31) for min/max."""
    return F.when(F.year(p).between(1901, 2099), p)

show([f"<h3>Setup</h3><p>Spark {esc(spark.version)} · pandas {esc(pd.__version__)} · numpy {esc(np.__version__)}</p>",
      tbl(pd.DataFrame([("FACT", FACT), ("SNAP_COL", SNAP_COL), ("SNAP_VALUE", SNAP_VALUE), ("ROW_KEY", ROW_KEY),
                        ("Decode entries", len(DECODE)), ("OUT_DIR", os.path.abspath(OUT_DIR))],
                       columns=["setting", "value"]))])


In [ ]:
# ============================================================
# 2. Inputs: fact snapshot and current dimension rows
# ============================================================
def flagv(c):
    return F.upper(F.trim(F.col(c).cast("string")))

def current_rows(table, key):
    """Current row per key: active, not purged, open-ended, newest effective / update date."""
    full = f"{CDM}.{table}"
    try:
        d = spark.table(full)
    except Exception as e:
        raise RuntimeError(f"Cannot read {full}: {str(e).splitlines()[0][:200]}")
    cols = d.columns
    if key not in cols:
        raise RuntimeError(f"{full} has no column {key}")
    order = []
    if "active_flg" in cols:
        order.append(F.when(flagv("active_flg").isin("Y", "1", "TRUE", "A"), 0).otherwise(1))
    if "purge_flg" in cols:
        order.append(F.when(flagv("purge_flg").isin("Y", "1", "TRUE"), 1).otherwise(0))
    if "end_dt" in cols:
        order.append(F.col("end_dt").desc_nulls_first())
    for c in ("eff_dt", "update_dt", "create_dt", "src_load_dt"):
        if c in cols:
            order.append(F.col(c).desc_nulls_last())
    w = Window.partitionBy(key).orderBy(*(order or [F.lit(1)]))
    st = d.agg(F.count(F.lit(1)).alias("rows"), F.countDistinct(key).alias("keys")).collect()[0]
    cur = d.withColumn("_rn", F.row_number().over(w)).filter("_rn = 1").drop("_rn")
    return cur, {"table": table, "key": key, "rows": st["rows"], "keys": st["keys"],
                 "keys with 2+ versions": st["rows"] - st["keys"], "version order": ", ".join(
                     c for c in ("active_flg", "purge_flg", "end_dt", "eff_dt", "update_dt", "create_dt", "src_load_dt")
                     if c in cols) or "(none)"}

# ---- fact snapshot ----
fact = spark.table(FACT)
fact_cols = sorted({c for e in DECODE for c in e[1]} | {SNAP_COL, ROW_KEY, "time_key"})
miss = [c for c in fact_cols if c not in fact.columns]
if miss:
    raise RuntimeError(f"{FACT} lacks columns {miss}")
measures = [m for m in MEASURES + ["pd_percent_1_mnth"] if m in fact.columns]
snap = fact.agg(F.max(SNAP_COL)).collect()[0][0] if SNAP_VALUE == "latest" else SNAP_VALUE
fs_raw = fact.filter(F.col(SNAP_COL) == F.lit(snap)).select(*fact_cols, *measures)
n_raw = fs_raw.count()
if n_raw == 0:
    raise RuntimeError(f"No rows in {FACT} for {SNAP_COL} = {snap}")
fs = fs_raw.dropDuplicates([ROW_KEY]).persist(StorageLevel.MEMORY_AND_DISK)
N = fs.count()

# ---- dimensions ----
dstats, DIMS, notes = [], {}, []
for label, fcols, table, key, want, sort_by in DECODE:
    cur, st = current_rows(table, key)
    have = [c for c in want if c in cur.columns]
    lost = [c for c in want if c not in cur.columns]
    if not have:   # fall back to the dimension's text columns
        have = [f.name for f in cur.schema.fields if isinstance(f.dataType, T.StringType) and f.name != key][:5]
    if st["keys"] > 200000:
        raise RuntimeError(f"{table} has {st['keys']:,} keys; too large to decode in pandas")
    pdf = cur.select(F.col(key).cast("string").alias("k"), *[F.col(c).cast("string").alias(c) for c in have]).toPandas()
    DIMS[label] = (pdf, have)
    st["label columns used"] = ", ".join(have)
    st["label columns missing"] = ", ".join(lost)
    dstats.append(st)
DSTATS = pd.DataFrame(dstats)
for c in ("rows", "keys", "keys with 2+ versions"):
    DSTATS[c] = DSTATS[c].map(lambda x: f"{x:,}")

show(["<h3>Inputs</h3>",
      tbl(pd.DataFrame([("Fact", FACT), ("Snapshot", f"{SNAP_COL} = {snap}"), ("Rows in snapshot", f"{n_raw:,}"),
                        (f"Rows after one per {ROW_KEY}", f"{N:,}"),
                        ("Measures summed", ", ".join(measures))], columns=["item", "value"])),
      "<h4>Dimensions (current version kept per key)</h4>", tbl(DSTATS)])


In [ ]:
# ============================================================
# 3. Decoded values on the snapshot
# ============================================================
def _num(s):
    return pd.to_numeric(s, errors="coerce")

parts = [f"<h3>Decoded values — {N:,} customers, {esc(SNAP_COL)} = {esc(snap)}</h3>",
         "<p class='note'>Counts are customers; share of all customers in brackets. Measures belong to the first "
         "key column of each table. In dimension = key found in the dimension (some dimensions carry a -1 row).</p>"]
for label, fcols, table, key, want, sort_by in DECODE:
    dim_pdf, labs = DIMS[label]
    base = None
    for j, c in enumerate(fcols):
        aggs = [F.count(F.lit(1)).alias(c)]
        if j == 0:
            aggs += [F.sum(F.col(m).cast("double")).alias(m) for m in measures if m != "pd_percent_1_mnth"]
            if c == "pd_key" and "pd_percent_1_mnth" in measures:
                aggs.append(F.avg(F.col("pd_percent_1_mnth").cast("double")).alias("fact pd % (avg)"))
        g = fs.groupBy(F.col(c).cast("string").alias("k")).agg(*aggs).toPandas()
        base = g if base is None else base.merge(g, on="k", how="outer")
    base[fcols] = base[fcols].fillna(0).astype("int64")
    t = base.merge(dim_pdf, on="k", how="left", indicator=True)
    t["in dim"] = (t.pop("_merge") == "both")

    # coverage of each key column
    cov = []
    for c in fcols:
        n_all = t[c].sum()
        n_in = t.loc[t["in dim"], c].sum()
        n_na = t.loc[~t["in dim"] & (t["k"].isna() | t["k"].isin(["-1"])), c].sum()
        cov.append({"fact column": c, "in dimension": fpct(n_in / n_all if n_all else None),
                    "-1 / null, not in dimension": fpct(n_na / n_all if n_all else None),
                    "other keys missing from dimension": fpct((n_all - n_na - n_in) / n_all if n_all else None),
                    "distinct values": f"{int((t[c] > 0).sum()):,}"})

    # order: by the dimension's own order column when given, else by customers
    if sort_by and sort_by in t.columns:
        t = t.assign(_s=_num(t[sort_by])).sort_values(["_s", fcols[0]], ascending=[True, False]).drop(columns="_s")
    else:
        t = t.sort_values(fcols[0], ascending=False)
    t.to_csv(os.path.join(OUT_DIR, re.sub(r"\W+", "_", label.lower()).strip("_") + ".csv"), index=False)

    shown = t.head(TOP_N).copy()
    if len(t) > TOP_N:
        rest = t.iloc[TOP_N:]
        other = {"k": f"other ({len(rest):,} values)"}
        other.update({c: rest[c].sum() for c in fcols})
        other.update({m: rest[m].sum() for m in measures if m in rest.columns})
        shown = pd.concat([shown, pd.DataFrame([other])], ignore_index=True)
    view = pd.DataFrame({"key": shown["k"].fillna("(null)")})
    for c in labs:
        view[c] = shown[c].fillna("")
    view["in dim"] = shown["in dim"].map(lambda v: "" if pd.isna(v) else ("yes" if v else "NO"))
    for c in fcols:
        tot = t[c].sum()
        view[c] = [f"{int(v):,} ({100 * v / tot:.1f}%)" if tot else "" for v in shown[c].fillna(0)]
    for m in measures:
        if m in shown.columns and m != "pd_percent_1_mnth":
            view[m] = shown[m].map(fnum)
    if "fact pd % (avg)" in shown.columns:
        view["fact pd % (avg)"] = shown["fact pd % (avg)"].map(lambda x: "" if pd.isna(x) else f"{x:.4g}")
    parts += [f"<h4>{esc(label)} — {esc(table)}.{esc(key)}</h4>", tbl(pd.DataFrame(cov)), tbl(view)]

parts.append(f"<p class='note'>Saved full tables to {esc(os.path.abspath(OUT_DIR))}</p>")
show(parts)


In [ ]:
# ============================================================
# 4. time_key: meaning and relation to as_of_dt
# ============================================================
td_cur, td_st = current_rows(TIME_DIM, "time_key")
dt_cols = [f.name for f in td_cur.schema.fields if isinstance(f.dataType, (T.DateType, T.TimestampType))]
dt_cols += [c for c in td_cur.columns if re.search(r"(_dt$|date)", c.lower()) and c not in dt_cols]
dt_cols = dt_cols[:8]

# snapshot window (as_of_dt is yyyymmdd); if it does not parse, all snapshots are used
try:
    _lo = int((_dt.datetime.strptime(str(snap), "%Y%m%d") - _dt.timedelta(days=TIME_LOOKBACK_DAYS)).strftime("%Y%m%d"))
    win = fact.filter(F.col(SNAP_COL) >= F.lit(_lo))
    win_note = f"{SNAP_COL} from {_lo} to {snap}"
except ValueError:
    win, win_note = fact, "all snapshots"
pairs = (win.groupBy(F.col(SNAP_COL).alias("snapshot"), F.col("time_key").cast("string").alias("time_key"))
            .count().toPandas())
pairs["s"] = _num(pairs["snapshot"])
pairs["t"] = _num(pairs["time_key"])
pairs = pairs.sort_values(["s", "t"])
keys_per_snap = pairs.groupby("snapshot")["time_key"].nunique()
snaps_per_key = pairs.groupby("time_key")["snapshot"].nunique()
mono = pairs.drop_duplicates("snapshot")["t"].is_monotonic_increasing
looks_ymd = pairs["time_key"].str.fullmatch(r"(19|20)\d{6}").mean()

checks = pd.DataFrame([
    ("Window", win_note),
    ("Snapshots / distinct time_keys", f"{pairs['snapshot'].nunique():,} / {pairs['time_key'].nunique():,}"),
    ("time_keys per snapshot (max)", int(keys_per_snap.max())),
    ("Snapshots per time_key (max)", int(snaps_per_key.max())),
    ("time_key rises with " + SNAP_COL, "yes" if mono else "NO"),
    ("time_key equals " + SNAP_COL, fpct((pairs["time_key"] == pairs["snapshot"].astype(str)).mean())),
    ("time_key looks like yyyymmdd", fpct(looks_ymd)),
    ("Time dimension rows / keys", f"{td_st['rows']:,} / {td_st['keys']:,}"),
], columns=["check", "value"])

last = pairs.tail(12)
tdp = (td_cur.filter(F.col("time_key").cast("string").isin(last["time_key"].unique().tolist()))
             .select(F.col("time_key").cast("string").alias("time_key"),
                     *[F.col(c).cast("string").alias(c) for c in dt_cols]).toPandas())
lastv = last[["snapshot", "time_key", "count"]].merge(tdp, on="time_key", how="left")
lastv["count"] = lastv["count"].map(lambda x: f"{x:,}")
lastv = lastv.rename(columns={"snapshot": SNAP_COL, "count": "rows"})
show(["<h3>time_key</h3>", tbl(checks),
      f"<h4>Latest 12 snapshots with time dimension dates ({esc(', '.join(dt_cols)) or 'no date columns found'})</h4>",
      tbl(lastv),
      "<p class='note'>If time_key rises with as_of_dt, the largest time_key is the latest period in tables that "
      "only carry time_key (the explore notebook's key mode relies on this).</p>"])


In [ ]:
# ============================================================
# 5. Instrument types: flag columns (WRRE and other exposure splits)
# ============================================================
it_cur, it_st = current_rows(INSTR_DIM, "instrument_type_dim_key")
it_cols = it_cur.columns
grp = next((c for c in ("instrument_type_group_desc", "instrument_group_cd", "instrument_type_desc") if c in it_cols), None)
flag_cols = [c for c in it_cols if re.search(r"(_flg|_ind)$", c.lower()) and c.lower() not in ("active_flg", "purge_flg")]
yes = ["Y", "1", "TRUE", "YES"]

rows = []
for c in flag_cols:
    vc = it_cur.groupBy(flagv(c).alias("v")).count().toPandas().sort_values("count", ascending=False)
    on = it_cur.filter(flagv(c).isin(*yes))
    n_on = on.count()
    ex = (on.groupBy(F.col(grp).cast("string").alias("g")).count().orderBy(F.col("count").desc()).limit(8).toPandas()
          if grp and n_on else pd.DataFrame(columns=["g", "count"]))
    rows.append({"flag column": c,
                 "values": "; ".join(f"{'(null)' if v is None else v} {n:,}" for v, n in zip(vc["v"], vc["count"])),
                 f"{grp or 'group'} where flag = Y (instrument types)": "; ".join(f"{g} ({n})" for g, n in zip(ex["g"], ex["count"]))})
it_pdf = it_cur.select(*[F.col(c).cast("string").alias(c) for c in it_cols]).toPandas()
it_pdf.to_csv(os.path.join(OUT_DIR, "instrument_types.csv"), index=False)

show(["<h3>Instrument types</h3>",
      f"<p>{esc(INSTR_DIM)}: {it_st['rows']:,} rows, {it_st['keys']:,} keys. Columns: {esc(', '.join(it_cols))}</p>",
      "<h4>Flag columns</h4>", tbl(pd.DataFrame(rows)),
      f"<p class='note'>Full dimension saved: {esc(os.path.abspath(os.path.join(OUT_DIR, 'instrument_types.csv')))}</p>"])
